# Task 11 Solutions — Exception Handling

This notebook solves every problem from Task 11 with complete, runnable code and the exact printed output. The whole sheet is one theme applied six ways: **make programs fail gracefully instead of crashing**. Q1 is the flagship exercise — a given function can raise several different errors, and the solution wraps calls in `try/except` with one specific handler per exception, plus `else` and `finally`, all *without altering the function*. Q2 shows nested exception handling in a tight loop: a mixed list of dicts, integers, and numeric strings is summed, and each `TypeError` is escalated to a fallback strategy (`AttributeError`, then `int()`). Q3 applies the discipline to file I/O, using the `else` block specifically for the success message. Q4 builds a guessing-game loop with three custom exception classes — `ValueTooLarge`, `ValueTooSmall`, `GuessError` — each carrying its own `display()` method. Q5 validates a name and age before letting someone cast a vote, with `InvalidName` and `InvalidAge` exceptions, an `else` for success, and a `finally` that always thanks the user. Q6 closes with a neat trick: an infinite loop that deliberately raises `StopIteration` after twenty numbers to exit.

Work through the solutions in order; each has a theory cell explaining the conceptual choice before you read the code. The recurring lessons are: catch **specific** exception types, order handlers from narrow to broad, put the happy path in `else`, and clean up in `finally`. These six programs are the complete practical vocabulary of exception handling in Python.

### `Q-1`: You are given a function definition. There might be several issues on execution of this function. You are asked to do exception handling for diffrent errors that this function goes in to `without altering this function`. And print error text.



Function parameters `l -> list, s -> could be anything`

```
def function(l: list, s, **args):
    last_element = l[-1]
    
    l[int(s)]=10
    any_element = l[int(s)+10]
    l[s]=10
    
    res = sum(l)
    
    p = args['p']
    # print(p)
    return res/last_element * p + any_element

```
Check for different function calls:-

```
function([1,2,1], 12)
function([1,2,1]*9, '1-2')
function([1,'2',1]*9, 12)
function([1,'2',1]*9, 12)
function([1,2,0]*9, 12  )
function([1,2,1]*9, 12, p=None)
function([1,2,0]*9, 12, p=10)
```

### Q1 — handling every error of a fixed function

The rule is stated in the question: we cannot touch `function`, we can only control how calls to it are protected. The tool is a `try` block wrapped around the call, followed by one `except` clause per exception the function might raise:

```python
try:
    res = function([1,2,1]*9, 12, p=10)
except IndexError as i:
    print(type(i)); print(i)
except ValueError as v:
    print(type(v)); print(v)
except TypeError as t:
    print(type(t)); print(t)
except KeyError as k:
    print(type(k)); print(k)
except ZeroDivisionError as z:
    print(type(z)); print(z)
else:
    print("Result:", res)
finally:
    print("Thank you")
```

Study what the function could raise: `l[-1]` with an empty list → `IndexError`; `l[int(s)]` where `s` is not numeric → `ValueError`; `l[s] = 10` with a non-integer `s` → `TypeError`; `args['p']` when no `p` is passed → `KeyError`; dividing by `last_element == 0` → `ZeroDivisionError`. The chosen call `function([1,2,1]*9, 12, p=10)` succeeds, so the `else` block prints `Result: 452.0` and `finally` prints `Thank you` regardless. Two habits to copy: print `type(e)` as well as the message, and always keep `finally` for guaranteed cleanup.

In [ ]:
# Write code here
def function(l: list, s, **args):
    last_element = l[-1]
    
    l[int(s)]=10
    any_element = l[int(s)+10]
    l[s]=10
    
    res = sum(l)
    
    p = args['p']
    # print(p)
    return res/last_element * p + any_element
try:
    res = function([1,2,1]*9, 12, p=10)
except IndexError as i:
    print(type(i))
    print(i)
except ValueError as v:
    print(type(v))
    print(v)
except TypeError as t:
    print(type(t))
    print(t)
except KeyError as k:
    print(type(k))
    print(k)
except ZeroDivisionError as z:
    print(type(z))
    print(z)
else:
    print("Result:", res)
finally:
    print("Thank you")

Result: 452.0
Thank you


### `Q-2:` You are given a code snippet. There might be several issues on execution of this code. You are asked to do exception handling for diffrent errors, condition is what ever happens we need to execute last line printing correct result of `sum of elements`.

List have elemnts as any no of  `key-pair dict with key as list index and value as any integer`, `integers` and `numeric-strings`. There is always only one element in the dict.


```
l = [{0:2},2,3,4,'5', {5:10}]
# For calculating sum of above list
s=0
for i in range(len(l)):
    #You can Edit code from here
    s += l[i].get(i)
    s += l[i]
    s += int(l[i])


print(s)
```

### Q2 — summing a heterogenous list with nested handlers

Every element of `l` is either a dict `{index: value}`, a plain integer, or a numeric string, and all three must contribute to one running sum. The strategy is sequential attempts with escalating fallbacks:

```python
for i in range(len(l)):
    try:
        s += l[i]
    except TypeError:
        try:
            s += l[i].get(i)
        except AttributeError:
            s += int(l[i])
```

Trace the logic per element. For an integer, `s += l[i]` works — no exception. For a string like `'5'`, adding to `s` raises `TypeError`, so the outer handler tries `l[i].get(i)`; strings have no `.get`, raising `AttributeError`, and the inner handler falls back to `int('5')`. For the dict `{0: 2}`, `s += l[i]` raises `TypeError` (dict + int is illegal), `.get(i)` returns the value under its own index (`d[0]` from the dict `{0:2}` is `2`), and no further fallback is needed. The sum runs `0 + 2 + 2 + 3 + 4 + 5 + 10 = 26`, printed exactly. The nested `try` is the elegant part: each `except` proposes a more compatible interpretation of the element.

In [ ]:
# Write code here
l = [{0:2},2,3,4,'5', {5:10}]
# For calculating sum of above list
s=0
for i in range(len(l)):
    try:
        s += l[i]
    except TypeError:
        try:
            s += l[i].get(i)
        except AttributeError:
            s += int(l[i])
print(s)

26


### `Q-3:`: File Handling with Exception handling

Write a program that opens a text file and write data to it as "Hello, Good Morning!!!". Handle exceptions that can be generated during the I/O operations. Do not show the success message on the main exception handling block (write inside the else block).

### Q3 — file I/O guarded, success message in else

The requirement is explicit: do not print the success message inside the exception block — that belongs in `else`. Statically:

```python
try:
    with open("text_file.txt", "w") as f:
        f.write("Hello, Good Morning!!!")
except IOError:
    print("Error working with file...")
else:
    print("File written successfully")
```

`open` and `write` are I/O operations, so the realistic failure is an `IOError` (permissions, disk errors, bad path). The `try` block protects exactly those calls. If the write succeeds, `else` — which runs only when no exception occurred — announces `File written successfully`; if an `IOError` is raised, the `except` prints the problem and the `else` stays silent. Note the `with` statement still closes the file automatically either way. This separation between normal completion (`else`) and failure handling (`except`) is what the question is testing: knowing *where* each message must live in the `try/except/else` structure.

In [ ]:
# write code here
try:
    with open("text_file.txt", "w") as f:
        f.write("Hello, Good Morning!!!")
except IOError:
    print("Error working with file...")
else:
    print("File written successfully")

File written successfully


### `Q-4`: Number game program.

Write a number game program. Ask the user to enter a number. If the number is greater than number to be guessed, raise a **ValueTooLarge** exception. If the value is smaller the number to be guessed the, raise a **ValueTooSmall** exception and prompt the user to enter again. Quit the program only when the user enters the correct number. Also raise **GuessError** if user guess a number less than 1.

### Q4 — custom exceptions power a guessing game

The game needs three error types the language does not provide, so we subclass `Exception` and give each class a `display()` method for its message:

```python
class ValueTooLarge(Exception):
    def display(self):
        print("Input value is too large")
class ValueTooSmall(Exception):
    def display(self):
        print("Input value is too small")
class GuessError(Exception):
    def display(self):
        print("Guess the number between 1 and 100")
```

The loop computes `num = random.randint(1, 100)` once, then keeps reading guesses. `guess < 1` raises `GuessError`; otherwise compare to `num` and raise `ValueTooSmall` or `ValueTooLarge`. The `try` block contains the full validation, and each `except` catches one custom class and calls its `display()` — so the loop continues until the guess equals `num`, at which point `print("Great You succeeded...")` and `break` end the game. Notice the power of raising: instead of `if/elif/else` printing messages, the *control flow* is driven by exceptions, and the message logic lives inside the exception classes — a clean separation that Q5 mirrors for voting.

In [ ]:
# Write code here
import random

class ValueTooLarge(Exception):
    def display(self):
        print("Input value is too large")

class ValueTooSmall(Exception):
    def display(self):
        print("Input value is too small")
    
class GuessError(Exception):
    def display(self):
        print("Guess the number between 1 and 100")

num = random.randint(1, 100)
while 1:
    try:
        guess = int(input("Enter a number:"))

        if guess < 1:
            raise GuessError

        if guess == num:
            print("Great You succeeded...")
            break
        if guess < num:
            raise ValueTooSmall
        if guess > num:
            raise ValueTooLarge
    except ValueTooSmall as s:
        s.display()
    except ValueTooLarge as l:
        l.display()
    except GuessError as g:
        g.display()


Enter a number: 5


Input value is too small


Enter a number: 1


Input value is too small


Enter a number: 99


...

Enter a number: 55


Input value is too small


Enter a number: 65


Input value is too large


KeyboardInterrupt: Interrupted by user

### `Q-5:` Cast vote

Write a program that validate name and age as entered by the user to determine whether the person can cast vote or not. To handle the age, create **InvalidAge** exception and for name, create **InvalidName** exception. The name will be invalid when the string will be empty or name has only one word.

Example 1:

Input:

```bash
Enter the name:               goransh singh
Enter the age: 25
```

Output:

```bash
Goransh Singh  Congratulation !!! You can vote.
```

### Q5 — validating name and age before casting a vote

This solution pairs two custom exceptions with the full `try/except/else/finally` structure:

```python
class InvalidAge(Exception):
    def display(self):
        print("Sorry!! Age cannot be below 18. Please come after that")
class InvalidName(Exception):
    def display(self):
        print("Please enter a valid name...")

try:
    name = input("Enter your name: ")
    if len(name) == 0 or len(name.split()) < 2:
        raise InvalidName
    age = int(input("Enter your age: "))
    if age < 18:
        raise InvalidAge
except InvalidName as i:
    i.display()
except InvalidAge as a:
    a.display()
else:
    print("Your vote is accepted successfully")
finally:
    print("Thank You")
```

The name fails if it is empty (`len(name) == 0`) or has only one word (`len(name.split()) < 2`) — the spec's definition of "invalid". The age fails if it is below 18; `int(input(...))` additionally raises a natural `ValueError` for non-numeric input, which propagates since we did not catch it. On success the `else` accepts the vote; `finally` prints `Thank You` no matter which branch ran. The sample run shows a valid name with two words and age 19 producing the accepted message plus the final thank-you — the exact structure the exercise wanted.

In [ ]:
# Write code here
class InvalidAge(Exception):
    def display(self):
        print("Sorry!! Age cannot be below 18. Please come after that")
class InvalidName(Exception):
    def display(self):
        print("Please enter a valid name...")
try:
    name = input("Enter your name: ")
    if len(name) == 0 or len(name.split()) < 2:
        raise InvalidName

    age = int(input("Enter your age: "))
    if age < 18:
        raise InvalidAge
except InvalidName as i:
    i.display()
except InvalidAge as a:
    a.display()
else:
    print("Your vote is accepted successfully")
finally:
    print("Thank You")

Enter your name:  Raj sdfsdf
Enter your age:  19


Your vote is accepted successfully
Thank You


### `Q-6`: Write a python function which infinitely prints natural numbers in a single line. Raise the **StopIteration** exception after displaying first 20 numnbers to exit from the program.

### Q6 — StopIteration to end an infinite loop

The goal is an infinite printer that exits after exactly twenty numbers by **raising** `StopIteration`:

```python
def display(n):
    i = 0
    while True:
        try:
            n += 1
            i += 1
            if i == 21:
                raise StopIteration
        except StopIteration:
            break
        else:
            print(n, end=' ')
```

Starting from `100`, each loop iteration increments the counter `i` and the number `n`. When `i` reaches 21, `raise StopIteration` fires *inside* the `try`, the `except StopIteration` handler catches it, `break` leaves the loop, and the `else` block — skipped whenever an exception occurred — means the 21st value is never printed. The `else` here is the elegant detail: it prints only the twenty successful iterations, so the output runs `101 102 ... 120` and stops. The trick extends what you learned about iterators, where `StopIteration` is normally raised internally to end a `for` loop — here we raise it ourselves to end a `while`. Use `end=' '` to keep the numbers on one line.

In [ ]:
# Write code here
def display(n):
    i=0
    while True:
        try:
            n+=1
            i+=1
            if i==21:
                raise StopIteration
        except StopIteration:
            break
        else:
            print(n, end =' ')
display(100)

101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 

## Summary

This sheet applied exception handling to six realistic programs. In Q1 you wrapped a fixed, error-prone function in `try/except` with a specific handler per exception, `else` for the happy path, and `finally` for guaranteed output — without touching the function. Q2 showed nested fallback handling, escalating from a direct add to `.get()` to `int()`, to sum a mixed list. Q3 confined an `IOError` and placed the success message correctly in `else`. Q4 built a guessing game driven by three custom exception classes, each with its own `display()`. Q5 validated voting eligibility with `InvalidName`/`InvalidAge`, using the full `try/except/else/finally` structure. Q6 ended an infinite loop by deliberately raising `StopIteration` after twenty numbers. The throughline: catch specific types, put normal completion in `else`, guarantee cleanup in `finally`, and let custom exceptions carry both meaning and behaviour.